In [24]:
# For tips on running notebooks in Google Colab, see
# https://pytorch.org/tutorials/beginner/colab
%matplotlib inline

Introduction to `torch.compile`
===============================

**Author:** William Wen


`torch.compile` is the latest method to speed up your PyTorch code!
`torch.compile` makes PyTorch code run faster by JIT-compiling PyTorch
code into optimized kernels, all while requiring minimal code changes.

In this tutorial, we cover basic `torch.compile` usage, and demonstrate
the advantages of `torch.compile` over previous PyTorch compiler
solutions, such as
[TorchScript](https://pytorch.org/docs/stable/jit.html) and [FX
Tracing](https://pytorch.org/docs/stable/fx.html#torch.fx.symbolic_trace).

**Contents**

::: {.contents local=""}
:::

**Required pip Dependencies**

-   `torch >= 2.0`
-   `torchvision`
-   `numpy`
-   `scipy`
-   `tabulate`

**System Requirements** - A C++ compiler, such as `g++` - Python
development package (`python-devel`/`python-dev`)


NOTE: a modern NVIDIA GPU (H100, A100, or V100) is recommended for this
tutorial in order to reproduce the speedup numbers shown below and
documented elsewhere.


In [25]:
import torch # type: ignore
import warnings

gpu_ok = False
if torch.cuda.is_available():
    device_cap = torch.cuda.get_device_capability()
    if device_cap in ((7, 0), (8, 0), (9, 0)):
        gpu_ok = True

if not gpu_ok:
    warnings.warn(
        "GPU is not NVIDIA V100, A100, or H100. Speedup numbers may be lower "
        "than expected."
    )

/tmp/ipykernel_2820526/2382151684.py:11: UserWarning: GPU is not NVIDIA V100, A100, or H100. Speedup numbers may be lower than expected.
  warnings.warn(


Basic Usage
===========

`torch.compile` is included in the latest PyTorch. Running TorchInductor
on GPU requires Triton, which is included with the PyTorch 2.0 nightly
binary. If Triton is still missing, try installing `torchtriton` via pip
(`pip install torchtriton --extra-index-url "https://download.pytorch.org/whl/nightly/cu117"`
for CUDA 11.7).

Arbitrary Python functions can be optimized by passing the callable to
`torch.compile`. We can then call the returned optimized function in
place of the original function.


In [26]:
def foo(x, y):
    a = torch.sin(x)
    b = torch.cos(y)
    return a + b
opt_foo1 = torch.compile(foo)
print(opt_foo1(torch.randn(10, 10), torch.randn(10, 10)))

tensor([[ 1.8473,  0.1378,  1.0981,  0.2742,  1.0346, -0.0764, -0.1089,  1.7458,
          1.9014,  1.0804],
        [ 0.7649,  1.1579,  0.6695,  0.6186,  1.8834,  0.7574,  0.9407,  0.7573,
          1.4029,  0.1887],
        [ 1.1217,  1.2604,  0.7001,  0.0790,  0.9063,  1.5919, -0.5565, -0.4514,
          1.1595, -0.6463],
        [ 0.1645,  1.4742,  0.7292, -0.2344,  1.6080,  1.2059, -1.3711,  0.9856,
          0.7278,  0.6951],
        [ 0.2446, -0.1625,  1.0092,  0.3561,  0.2269,  1.6192, -0.0386,  1.2513,
          0.7495,  0.8011],
        [-0.0877, -0.2777,  1.8492,  1.1605,  1.3417,  1.1661,  0.5051,  1.8985,
          1.5237,  1.2689],
        [ 0.8547, -0.2986,  0.9911,  1.9416, -0.4943,  0.4861, -0.5233,  0.6411,
         -0.1466,  0.0030],
        [ 1.4088, -0.3188,  1.5730, -0.2099,  1.0971,  0.5727,  0.0230, -0.0613,
          0.1305, -0.1430],
        [ 0.5958,  1.0610,  0.0159,  0.4738,  1.7620,  0.2256,  0.5863,  0.4258,
         -0.2040,  1.2508],
        [ 0.5131,  

Alternatively, we can decorate the function.


In [27]:
t1 = torch.randn(10, 10)
t2 = torch.randn(10, 10)

@torch.compile
def opt_foo2(x, y):
    a = torch.sin(x)
    b = torch.cos(y)
    return a + b
print(opt_foo2(t1, t2))

tensor([[ 1.1154,  0.6048, -0.9178,  0.9318,  1.7118,  1.2974,  0.8674,  1.6500,
         -0.3786, -0.0545],
        [ 1.9129, -0.0584,  1.3736,  0.9822,  1.7186,  0.6403,  0.8960,  1.0243,
          0.7271, -0.7148],
        [-1.8399,  0.2338, -0.4416, -0.1103,  0.0089,  1.2754,  0.4850,  1.6755,
          0.9111,  0.6378],
        [-0.2385,  0.2293,  1.2210,  0.3471, -1.1272,  0.7139, -0.2995,  0.2635,
          0.5182,  1.0863],
        [ 0.3423,  0.1132,  1.6149, -0.4491,  1.3921,  1.8853,  1.4132, -0.5776,
          1.1053,  0.0266],
        [ 0.8704, -0.3149, -0.8969,  1.1323,  1.5411,  1.9151,  1.5259,  1.4132,
         -0.3483,  0.6434],
        [-0.4165,  0.0675,  0.2250, -0.1822, -0.5666, -0.1739,  1.6506,  1.3526,
          0.7882,  0.7300],
        [-0.1377,  1.3708, -0.1270,  0.8106, -0.5364, -0.9421,  0.8734,  1.2622,
          0.0903,  0.2570],
        [-0.2054,  0.3534,  0.8668,  0.6759,  0.2747,  0.3690,  0.7129, -1.2504,
          1.1926,  0.1488],
        [ 0.5763, -

We can also optimize `torch.nn.Module` instances.


In [28]:
t = torch.randn(10, 100)

class MyModule(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = torch.nn.Linear(100, 10)

    def forward(self, x):
        return torch.nn.functional.relu(self.lin(x))

mod = MyModule()
mod.compile()
print(mod(t))
## or:
# opt_mod = torch.compile(mod)
# print(opt_mod(t))

tensor([[0.0000, 0.0000, 0.2455, 0.0000, 0.2804, 0.0000, 0.5240, 0.0000, 0.0000,
         0.0000],
        [0.8931, 0.0000, 0.0000, 0.0428, 0.0000, 0.0000, 0.0000, 0.0000, 0.1228,
         0.1007],
        [0.5343, 0.3854, 0.4857, 0.0000, 0.3012, 0.0000, 0.0000, 0.6998, 0.8460,
         0.0000],
        [0.4727, 0.0000, 0.2977, 1.0194, 0.0000, 0.3366, 0.0699, 0.0000, 0.0873,
         0.5277],
        [0.5375, 0.0000, 0.0000, 0.3499, 0.0000, 0.0000, 0.7840, 0.0000, 0.7856,
         0.0000],
        [0.0366, 0.3849, 0.7366, 0.0000, 0.4224, 0.0000, 0.7349, 0.4581, 0.0000,
         0.0000],
        [0.0000, 0.8551, 0.8205, 0.0000, 0.0000, 0.0000, 0.6691, 0.2027, 0.3060,
         0.7667],
        [0.0000, 0.1240, 0.3557, 0.0000, 0.0000, 0.0000, 1.0689, 0.1331, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.7611, 0.0000, 0.4995, 0.2299, 0.1252, 0.8134, 0.0000,
         0.4500],
        [0.4799, 0.0000, 0.7548, 0.6452, 0.6079, 0.0000, 0.6039, 0.3126, 0.7008,
         0.0000]], grad_fn=<

torch.compile and Nested Calls
==============================

Nested function calls within the decorated function will also be
compiled.


In [29]:
def nested_function(x):
    return torch.sin(x)

@torch.compile
def outer_function(x, y):
    a = nested_function(x)
    b = torch.cos(y)
    return a + b

print(outer_function(t1, t2))

tensor([[ 1.1154,  0.6048, -0.9178,  0.9318,  1.7118,  1.2974,  0.8674,  1.6500,
         -0.3786, -0.0545],
        [ 1.9129, -0.0584,  1.3736,  0.9822,  1.7186,  0.6403,  0.8960,  1.0243,
          0.7271, -0.7148],
        [-1.8399,  0.2338, -0.4416, -0.1103,  0.0089,  1.2754,  0.4850,  1.6755,
          0.9111,  0.6378],
        [-0.2385,  0.2293,  1.2210,  0.3471, -1.1272,  0.7139, -0.2995,  0.2635,
          0.5182,  1.0863],
        [ 0.3423,  0.1132,  1.6149, -0.4491,  1.3921,  1.8853,  1.4132, -0.5776,
          1.1053,  0.0266],
        [ 0.8704, -0.3149, -0.8969,  1.1323,  1.5411,  1.9151,  1.5259,  1.4132,
         -0.3483,  0.6434],
        [-0.4165,  0.0675,  0.2250, -0.1822, -0.5666, -0.1739,  1.6506,  1.3526,
          0.7882,  0.7300],
        [-0.1377,  1.3708, -0.1270,  0.8106, -0.5364, -0.9421,  0.8734,  1.2622,
          0.0903,  0.2570],
        [-0.2054,  0.3534,  0.8668,  0.6759,  0.2747,  0.3690,  0.7129, -1.2504,
          1.1926,  0.1488],
        [ 0.5763, -

In the same fashion, when compiling a module all sub-modules and methods
within it, that are not in a skip list, are also compiled.


In [30]:
class OuterModule(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.inner_module = MyModule()
        self.outer_lin = torch.nn.Linear(10, 2)

    def forward(self, x):
        x = self.inner_module(x)
        return torch.nn.functional.relu(self.outer_lin(x))

outer_mod = OuterModule()
outer_mod.compile()
print(outer_mod(t))

tensor([[0.1746, 0.0000],
        [0.6827, 0.0000],
        [0.2588, 0.0000],
        [0.6488, 0.0000],
        [0.1535, 0.0000],
        [0.2966, 0.0000],
        [0.4712, 0.0000],
        [0.2149, 0.0000],
        [0.2603, 0.0000],
        [0.2457, 0.0000]], grad_fn=<CompiledFunctionBackward>)


We can also disable some functions from being compiled by using
`torch.compiler.disable`. Suppose you want to disable the tracing on
just the `complex_function` function, but want to continue the tracing
back in `complex_conjugate`. In this case, you can use
`torch.compiler.disable(recursive=False)` option. Otherwise, the default
is `recursive=True`.


In [31]:
def complex_conjugate(z):
    return torch.conj(z)

@torch.compiler.disable(recursive=False)
def complex_function(real, imag):
    # Assuming this function cause problems in the compilation
    z = torch.complex(real, imag)
    return complex_conjugate(z)

def outer_function():
    real = torch.tensor([2, 3], dtype=torch.float32)
    imag = torch.tensor([4, 5], dtype=torch.float32)
    z = complex_function(real, imag)
    return torch.abs(z)

# Try to compile the outer_function
try:
    opt_outer_function = torch.compile(outer_function)
    print(opt_outer_function())
except Exception as e:
    print("Compilation of outer_function failed:", e)

tensor([4.4721, 5.8310])


Best Practices and Recommendations
==================================

Behavior of `torch.compile` with Nested Modules and Function Calls

When you use `torch.compile`, the compiler will try to recursively
compile every function call inside the target function or module inside
the target function or module that is not in a skip list (such as
built-ins, some functions in the torch.\* namespace).

**Best Practices:**

1\. **Top-Level Compilation:** One approach is to compile at the highest
level possible (i.e., when the top-level module is initialized/called)
and selectively disable compilation when encountering excessive graph
breaks or errors. If there are still many compile issues, compile
individual subcomponents instead.

2\. **Modular Testing:** Test individual functions and modules with
`torch.compile` before integrating them into larger models to isolate
potential issues.

3\. **Disable Compilation Selectively:** If certain functions or
sub-modules cannot be handled by [torch.compile]{.title-ref}, use the
[torch.compiler.disable]{.title-ref} context managers to recursively
exclude them from compilation.

4\. **Compile Leaf Functions First:** In complex models with multiple
nested functions and modules, start by compiling the leaf functions or
modules first. For more information see [TorchDynamo APIs for
fine-grained
tracing](https://pytorch.org/docs/stable/torch.compiler_fine_grain_apis.html).

5.  **Prefer \`\`mod.compile()\`\` over \`\`torch.compile(mod)\`\`:**
    Avoids `_orig_` prefix issues in `state_dict`.

6\. **Use \`\`fullgraph=True\`\` to catch graph breaks:** Helps ensure
end-to-end compilation, maximizing speedup and compatibility with
`torch.export`.


Demonstrating Speedups
======================

Let\'s now demonstrate that using `torch.compile` can speed up real
models. We will compare standard eager mode and `torch.compile` by
evaluating and training a `torchvision` model on random data.

Before we start, we need to define some utility functions.


In [32]:
# Returns the result of running `fn()` and the time it took for `fn()` to run,
# in seconds. We use CUDA events and synchronization for the most accurate
# measurements.
def timed(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record() # type: ignore
    result = fn()
    end.record() # type: ignore
    torch.cuda.synchronize() # type: ignore
    return result, start.elapsed_time(end) / 1000

# Generates random input and targets data for the model, where `b` is
# batch size.
def generate_data(b):
    return (
        torch.randn(b, 3, 128, 128).to(torch.float32).cuda(),
        torch.randint(1000, (b,)).cuda(),
    )

N_ITERS = 10

from torchvision.models import densenet121
def init_model():
    return densenet121().to(torch.float32).cuda()

First, let\'s compare inference.

Note that in the call to `torch.compile`, we have the additional `mode`
argument, which we will discuss below.


In [33]:
model = init_model()

# Reset since we are using a different mode.
import torch._dynamo
torch._dynamo.reset()

model_opt = torch.compile(model, mode="reduce-overhead")

inp = generate_data(16)[0]
with torch.no_grad():
    print("eager:", timed(lambda: model(inp))[1])
    print("compile:", timed(lambda: model_opt(inp))[1])

eager: 0.0055838398933410645
compile: 5.90143896484375


Notice that `torch.compile` takes a lot longer to complete compared to
eager. This is because `torch.compile` compiles the model into optimized
kernels as it executes. In our example, the structure of the model
doesn\'t change, and so recompilation is not needed. So if we run our
optimized model several more times, we should see a significant
improvement compared to eager.


In [34]:
eager_times = []
for i in range(N_ITERS):
    inp = generate_data(16)[0]
    with torch.no_grad():
        _, eager_time = timed(lambda: model(inp))
    eager_times.append(eager_time)
    print(f"eager eval time {i}: {eager_time}")

print("~" * 10)

compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)[0]
    with torch.no_grad():
        _, compile_time = timed(lambda: model_opt(inp))
    compile_times.append(compile_time)
    print(f"compile eval time {i}: {compile_time}")
print("~" * 10)

import numpy as np
eager_med = np.median(eager_times)
compile_med = np.median(compile_times)
speedup = eager_med / compile_med
# assert(speedup > 1)
print(f"(eval) eager median: {eager_med}, compile median: {compile_med}, speedup: {speedup}x")
print("~" * 10)

eager eval time 0: 0.0055797438621521
eager eval time 1: 0.005381120204925537
eager eval time 2: 0.0053678078651428224
eager eval time 3: 0.00539788818359375
eager eval time 4: 0.0053788480758666995
eager eval time 5: 0.005362559795379638
eager eval time 6: 0.0053534722328186036
eager eval time 7: 0.005378975868225098
eager eval time 8: 0.005344255924224854
eager eval time 9: 0.005380959987640381
~~~~~~~~~~
compile eval time 0: 1.1450133056640626
compile eval time 1: 0.0068544321060180664
compile eval time 2: 0.0069621758460998535
compile eval time 3: 0.006445759773254394
compile eval time 4: 0.006386688232421875
compile eval time 5: 0.006390592098236084
compile eval time 6: 0.006456319808959961
compile eval time 7: 0.00643174409866333
compile eval time 8: 0.00638156795501709
compile eval time 9: 0.006389503955841065
~~~~~~~~~~
(eval) eager median: 0.005378911972045899, compile median: 0.006438751935958862, speedup: 0.8353966771115975x
~~~~~~~~~~


And indeed, we can see that running our model with `torch.compile`
results in a significant speedup. Speedup mainly comes from reducing
Python overhead and GPU read/writes, and so the observed speedup may
vary on factors such as model architecture and batch size. For example,
if a model\'s architecture is simple and the amount of data is large,
then the bottleneck would be GPU compute and the observed speedup may be
less significant.

You may also see different speedup results depending on the chosen
`mode` argument. The `"reduce-overhead"` mode uses CUDA graphs to
further reduce the overhead of Python. For your own models, you may need
to experiment with different modes to maximize speedup. You can read
more about modes
[here](https://pytorch.org/get-started/pytorch-2.0/#user-experience).

You may might also notice that the second time we run our model with
`torch.compile` is significantly slower than the other runs, although it
is much faster than the first run. This is because the
`"reduce-overhead"` mode runs a few warm-up iterations for CUDA graphs.

For general PyTorch benchmarking, you can try using
`torch.utils.benchmark` instead of the `timed` function we defined
above. We wrote our own timing function in this tutorial to show
`torch.compile`\'s compilation latency.

Now, let\'s consider comparing training.


In [35]:
model = init_model()
opt = torch.optim.Adam(model.parameters())

def train(mod, data):
    opt.zero_grad(True)
    pred = mod(data[0])
    loss = torch.nn.CrossEntropyLoss()(pred, data[1])
    loss.backward()
    opt.step()

eager_times = []
for i in range(N_ITERS):
    inp = generate_data(16)
    _, eager_time = timed(lambda: train(model, inp))
    eager_times.append(eager_time)
    print(f"eager train time {i}: {eager_time}")
print("~" * 10)

model = init_model()
opt = torch.optim.Adam(model.parameters())
train_opt = torch.compile(train, mode="reduce-overhead")

compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)
    _, compile_time = timed(lambda: train_opt(model, inp))
    compile_times.append(compile_time)
    print(f"compile train time {i}: {compile_time}")
print("~" * 10)

eager_med = np.median(eager_times)
compile_med = np.median(compile_times)
speedup = eager_med / compile_med
assert(speedup > 1)
print(f"(train) eager median: {eager_med}, compile median: {compile_med}, speedup: {speedup}x")
print("~" * 10)

eager train time 0: 0.025399295806884766
eager train time 1: 0.021426111221313476
eager train time 2: 0.020530048370361327
eager train time 3: 0.020506559371948244
eager train time 4: 0.020494335174560546
eager train time 5: 0.020508895874023436
eager train time 6: 0.020477951049804686
eager train time 7: 0.020542463302612304
eager train time 8: 0.020282367706298828
eager train time 9: 0.018951168060302736
~~~~~~~~~~
compile train time 0: 28.4638046875
compile train time 1: 5.10893896484375
compile train time 2: 0.018672544479370116
compile train time 3: 0.018050048828125
compile train time 4: 0.017572864532470703
compile train time 5: 0.017530624389648437
compile train time 6: 0.017497695922851563
compile train time 7: 0.017526784896850587
compile train time 8: 0.017002431869506836
compile train time 9: 0.015920960426330566
~~~~~~~~~~
(train) eager median: 0.02050772762298584, compile median: 0.01755174446105957, speedup: 1.1684153485987923x
~~~~~~~~~~


Again, we can see that `torch.compile` takes longer in the first
iteration, as it must compile the model, but in subsequent iterations,
we see significant speedups compared to eager.

We remark that the speedup numbers presented in this tutorial are for
demonstration purposes only. Official speedup values can be seen at the
[TorchInductor performance
dashboard](https://hud.pytorch.org/benchmark/compilers).


Comparison to TorchScript and FX Tracing
========================================

We have seen that `torch.compile` can speed up PyTorch code. Why else
should we use `torch.compile` over existing PyTorch compiler solutions,
such as TorchScript or FX Tracing? Primarily, the advantage of
`torch.compile` lies in its ability to handle arbitrary Python code with
minimal changes to existing code.

One case that `torch.compile` can handle that other compiler solutions
struggle with is data-dependent control flow (the `if x.sum() < 0:` line
below).


In [36]:
def f1(x, y):
    if x.sum() < 0:
        return -y
    return y

# Test that `fn1` and `fn2` return the same result, given
# the same arguments `args`. Typically, `fn1` will be an eager function
# while `fn2` will be a compiled function (torch.compile, TorchScript, or FX graph).
def test_fns(fn1, fn2, args):
    out1 = fn1(*args)
    out2 = fn2(*args)
    return torch.allclose(out1, out2)

inp1 = torch.randn(5, 5)
inp2 = torch.randn(5, 5)

TorchScript tracing `f1` results in silently incorrect results, since
only the actual control flow path is traced.


In [37]:
traced_f1 = torch.jit.trace(f1, (inp1, inp2))
print("traced 1, 1:", test_fns(f1, traced_f1, (inp1, inp2)))
print("traced 1, 2:", test_fns(f1, traced_f1, (-inp1, inp2)))

traced 1, 1: True
traced 1, 2: False


/tmp/ipykernel_2820526/4005623017.py:2: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.sum() < 0:


FX tracing `f1` results in an error due to the presence of
data-dependent control flow.


In [38]:
import traceback as tb
try:
    torch.fx.symbolic_trace(f1)
except:
    tb.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_2820526/1451191637.py", line 3, in <module>
    torch.fx.symbolic_trace(f1)
  File "/data/projects/rosellm/.conda/lib/python3.11/site-packages/torch/fx/_symbolic_trace.py", line 1307, in symbolic_trace
    graph = tracer.trace(root, concrete_args)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/projects/rosellm/.conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py", line 745, in _fn
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/data/projects/rosellm/.conda/lib/python3.11/site-packages/torch/fx/_symbolic_trace.py", line 843, in trace
    (self.create_arg(fn(*args)),),
                     ^^^^^^^^^
  File "/tmp/ipykernel_2820526/4005623017.py", line 2, in f1
    if x.sum() < 0:
       ^^^^^^^^^^^
  File "/data/projects/rosellm/.conda/lib/python3.11/site-packages/torch/fx/proxy.py", line 549, in __bool__
    return self.tracer.to_bool(self)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File 

If we provide a value for `x` as we try to FX trace `f1`, then we run
into the same problem as TorchScript tracing, as the data-dependent
control flow is removed in the traced function.


In [39]:
fx_f1 = torch.fx.symbolic_trace(f1, concrete_args={"x": inp1})
print("fx 1, 1:", test_fns(f1, fx_f1, (inp1, inp2)))
print("fx 1, 2:", test_fns(f1, fx_f1, (-inp1, inp2)))

fx 1, 1: True
fx 1, 2: False


/data/projects/rosellm/.conda/lib/python3.11/site-packages/torch/fx/_symbolic_trace.py:906: UserWarning: Was not able to add assertion to guarantee correct input x to specialized function. It is up to the user to make sure that your inputs match the inputs you specialized the function with.
  warnings.warn(


Now we can see that `torch.compile` correctly handles data-dependent
control flow.


In [40]:
# Reset since we are using a different mode.
torch._dynamo.reset()

compile_f1 = torch.compile(f1)
print("compile 1, 1:", test_fns(f1, compile_f1, (inp1, inp2)))
print("compile 1, 2:", test_fns(f1, compile_f1, (-inp1, inp2)))
print("~" * 10)

compile 1, 1: True
compile 1, 2: True
~~~~~~~~~~


TorchScript scripting can handle data-dependent control flow, but this
solution comes with its own set of problems. Namely, TorchScript
scripting can require major code changes and will raise errors when
unsupported Python is used.

In the example below, we forget TorchScript type annotations and we
receive a TorchScript error because the input type for argument `y`, an
`int`, does not match with the default argument type, `torch.Tensor`.


In [41]:
def f2(x, y):
    return x + y

inp1 = torch.randn(5, 5)
inp2 = 3

script_f2 = torch.jit.script(f2)
try:
    script_f2(inp1, inp2)
except:
    tb.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_2820526/895162831.py", line 9, in <module>
    script_f2(inp1, inp2)
RuntimeError: f2() Expected a value of type 'Tensor (inferred)' for argument 'y' but instead found type 'int'.
Inferred 'y' to be of type 'Tensor' because it was not annotated with an explicit type.
Position: 1
Value: 3
Declaration: f2(Tensor x, Tensor y) -> Tensor
Cast error details: Unable to cast 3 to Tensor


However, `torch.compile` is easily able to handle `f2`.


In [42]:
compile_f2 = torch.compile(f2)
print("compile 2:", test_fns(f2, compile_f2, (inp1, inp2)))
print("~" * 10)

compile 2: True
~~~~~~~~~~


Another case that `torch.compile` handles well compared to previous
compilers solutions is the usage of non-PyTorch functions.


In [43]:
import scipy
def f3(x):
    x = x * 2
    x = scipy.fft.dct(x.numpy())
    x = torch.from_numpy(x)
    x = x * 2
    return x

TorchScript tracing treats results from non-PyTorch function calls as
constants, and so our results can be silently wrong.


In [44]:
inp1 = torch.randn(5, 5)
inp2 = torch.randn(5, 5)
traced_f3 = torch.jit.trace(f3, (inp1,))
print("traced 3:", test_fns(f3, traced_f3, (inp2,)))

traced 3: False


/tmp/ipykernel_2820526/123398134.py:4: TracerWarning: Converting a tensor to a NumPy array might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  x = scipy.fft.dct(x.numpy())
/tmp/ipykernel_2820526/123398134.py:5: TracerWarning: torch.from_numpy results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  x = torch.from_numpy(x)


TorchScript scripting and FX tracing disallow non-PyTorch function
calls.


In [45]:
try:
    torch.jit.script(f3)
except:
    tb.print_exc()

try:
    torch.fx.symbolic_trace(f3)
except:
    tb.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_2820526/1991772511.py", line 2, in <module>
    torch.jit.script(f3)
  File "/data/projects/rosellm/.conda/lib/python3.11/site-packages/torch/jit/_script.py", line 1429, in script
    ret = _script_impl(
          ^^^^^^^^^^^^^
  File "/data/projects/rosellm/.conda/lib/python3.11/site-packages/torch/jit/_script.py", line 1205, in _script_impl
    fn = torch._C._jit_script_compile(
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/projects/rosellm/.conda/lib/python3.11/site-packages/torch/_jit_internal.py", line 1239, in _try_get_dispatched_fn
    return boolean_dispatched.get(fn)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/projects/rosellm/.conda/lib/python3.11/weakref.py", line 452, in get
    return self.data.get(ref(key),default)
                         ^^^^^^^^
TypeError: cannot create weak reference to 'uarray._Function' object
Traceback (most recent call last):
  File "/tmp/ipykernel_2820526/1991772511.py", lin

In comparison, `torch.compile` is easily able to handle the non-PyTorch
function call.


In [46]:
compile_f3 = torch.compile(f3)
print("compile 3:", test_fns(f3, compile_f3, (inp2,)))

/data/projects/rosellm/.conda/lib/python3.11/site-packages/torch/_dynamo/variables/functions.py:679: UserWarning: Graph break due to unsupported builtin scipy.fft._pocketfft.pypocketfft.PyCapsule.dct. This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind). If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround. If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use torch.compiler.allow_in_graph.
  torch._dynamo.utils.warn_once(msg)


InternalTorchDynamoError: AttributeError: 'str' object has no attribute 'IF_NEEDED'

from user code:
   File "/data/projects/rosellm/.conda/lib/python3.11/site-packages/scipy/fft/_realtransforms_backend.py", line 15, in torch_dynamo_resume_in__execute_at_12
    return xp.asarray(y)
  File "/data/projects/rosellm/.conda/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py", line 116, in asarray
    return np.array(obj, copy=copy, dtype=dtype, **kwargs)

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True


TorchDynamo and FX Graphs
=========================

One important component of `torch.compile` is TorchDynamo. TorchDynamo
is responsible for JIT compiling arbitrary Python code into [FX
graphs](https://pytorch.org/docs/stable/fx.html#torch.fx.Graph), which
can then be further optimized. TorchDynamo extracts FX graphs by
analyzing Python bytecode during runtime and detecting calls to PyTorch
operations.

Normally, TorchInductor, another component of `torch.compile`, further
compiles the FX graphs into optimized kernels, but TorchDynamo allows
for different backends to be used. In order to inspect the FX graphs
that TorchDynamo outputs, let us create a custom backend that outputs
the FX graph and simply returns the graph\'s unoptimized forward method.


In [ ]:
from typing import List
def custom_backend(gm: torch.fx.GraphModule, example_inputs: List[torch.Tensor]):
    print("custom backend called with FX graph:")
    gm.graph.print_tabular()
    return gm.forward

# Reset since we are using a different backend.
torch._dynamo.reset()

opt_model = torch.compile(init_model(), backend=custom_backend)
opt_model(generate_data(16)[0])

Using our custom backend, we can now see how TorchDynamo is able to
handle data-dependent control flow. Consider the function below, where
the line `if b.sum() < 0` is the source of data-dependent control flow.


In [ ]:
def bar(a, b):
    x = a / (torch.abs(a) + 1)
    if b.sum() < 0:
        b = b * -1
    return x * b

opt_bar = torch.compile(bar, backend=custom_backend)
inp1 = torch.randn(10)
inp2 = torch.randn(10)
opt_bar(inp1, inp2)
opt_bar(inp1, -inp2)

The output reveals that TorchDynamo extracted 3 different FX graphs
corresponding the following code (order may differ from the output
above):

1.  `x = a / (torch.abs(a) + 1)`
2.  `b = b * -1; return x * b`
3.  `return x * b`

When TorchDynamo encounters unsupported Python features, such as
data-dependent control flow, it breaks the computation graph, lets the
default Python interpreter handle the unsupported code, then resumes
capturing the graph.

Let\'s investigate by example how TorchDynamo would step through `bar`.
If `b.sum() < 0`, then TorchDynamo would run graph 1, let Python
determine the result of the conditional, then run graph 2. On the other
hand, if `not b.sum() < 0`, then TorchDynamo would run graph 1, let
Python determine the result of the conditional, then run graph 3.

This highlights a major difference between TorchDynamo and previous
PyTorch compiler solutions. When encountering unsupported Python
features, previous solutions either raise an error or silently fail.
TorchDynamo, on the other hand, will break the computation graph.

We can see where TorchDynamo breaks the graph by using
`torch._dynamo.explain`:


In [ ]:
# Reset since we are using a different backend.
torch._dynamo.reset()
explain_output = torch._dynamo.explain(bar)(torch.randn(10), torch.randn(10))
print(explain_output)

In order to maximize speedup, graph breaks should be limited. We can
force TorchDynamo to raise an error upon the first graph break
encountered by using `fullgraph=True`:


In [ ]:
opt_bar = torch.compile(bar, fullgraph=True)
try:
    opt_bar(torch.randn(10), torch.randn(10))
except:
    tb.print_exc()

And below, we demonstrate that TorchDynamo does not break the graph on
the model we used above for demonstrating speedups.


In [ ]:
opt_model = torch.compile(init_model(), fullgraph=True)
print(opt_model(generate_data(16)[0]))

We can use `torch.export` (from PyTorch 2.1+) to extract a single,
exportable FX graph from the input PyTorch program. The exported graph
is intended to be run on different (i.e. Python-less) environments. One
important restriction is that the `torch.export` does not support graph
breaks. Please check [this
tutorial](https://pytorch.org/tutorials/intermediate/torch_export_tutorial.html)
for more details on `torch.export`.


Conclusion
==========

In this tutorial, we introduced `torch.compile` by covering basic usage,
demonstrating speedups over eager mode, comparing to previous PyTorch
compiler solutions, and briefly investigating TorchDynamo and its
interactions with FX graphs. We hope that you will give `torch.compile`
a try!
